In [2]:
import os
import random
import datetime
import warnings
import numpy as np
import scipy.io as spio

def generateRescaleValue(inputMean):
    currVal = inputMean
    count = 0
    while currVal > 1:
        currVal = currVal/10
        count += 1
    return 10**count

def parseAllCovMat(inputCovMat):
    outputMat = 0
    for a,singleArr in enumerate(inputCovMat):
        rescaleValue = generateRescaleValue(np.abs(np.mean(singleArr)))
        if len(np.shape(singleArr)) == 3:
            if np.shape(outputMat) == ():
                outputMat = singleArr.astype(float)/rescaleValue
            else:
                outputMat = np.concatenate((outputMat,singleArr.astype(float)/rescaleValue),axis=0)
    meanCovMat = np.mean(np.abs(outputMat).astype(float),axis=0)
    lowerTriangleInds = np.tril_indices(np.shape(meanCovMat)[0])
    meanCovMat[lowerTriangleInds] = 0
    return meanCovMat+np.transpose(meanCovMat)

def parseUnitsByRegion(inputArr):
    pfcI = np.nonzero(inputArr-1)[0].flatten()
    hpcI = np.delete(np.arange(0,len(inputArr),1),pfcI,axis=0)
    return hpcI,pfcI

def generateDirectedGraphsSpatialOnly(name):
    spaFile = ''
    regFile = ''
    smat = spio.loadmat(spaFile)
    rmat = spio.loadmat(regFile)
    sScale = smat.get('scale')[0][0]
    spaConnection = parseAllCovMat(smat.get('allCovMat')[0]).astype(float)
    iHPC,iPFC = parseUnitsByRegion(rmat.get('regionIndex').flatten())
    return spaConnection,iHPC,iPFC

def generateDirectedGraphsCueOnly(name):
    cueFile = ''
    regFile = ''
    cmat = spio.loadmat(cueFile)
    rmat = spio.loadmat(regFile)
    cScale = cmat.get('scale')[0][0]
    cueConnection = parseAllCovMat(cmat.get('allCovMat')[0]).astype(float)
    iHPC,iPFC = parseUnitsByRegion(rmat.get('regionIndex').flatten())
    return cueConnection,iHPC,iPFC

def generateStepSizing(inputConnections):
    thresholdTrue = np.shape(inputConnections)[0]*np.mean(inputConnections)
    thresholdRange = np.arange(1.8*thresholdTrue,2.3*thresholdTrue,.1*thresholdTrue)
    remainRange = np.arange(.6,1.1,.1)
    return thresholdRange,remainRange

def loadValuesFromFileCue(fileOfInterest,justFileName):
    cueConnectionGraph,hpcInds,pfcInds = generateDirectedGraphsCueOnly(justFileName)    
    connDict = {'connectionAll':cueConnectionGraph,
                'connectionHPC':np.take(np.take(cueConnectionGraph,hpcInds,axis=1),hpcInds,axis=0),
                'connectionPFC':np.take(np.take(cueConnectionGraph,pfcInds,axis=1),pfcInds,axis=0),
               }
    rangeThresholdAll,rangeRemainAll = generateStepSizing(connDict['connectionAll'])
    rangeThresholdHPC,rangeRemainHPC = generateStepSizing(connDict['connectionHPC'])
    rangeThresholdPFC,rangeRemainPFC = generateStepSizing(connDict['connectionPFC'])
    rmat = spio.loadmat(fileOfInterest)
    cueErrorAll = np.abs(rmat.get('cueSpikeTrainError')-1)
    cueErrorHPC = np.abs(rmat.get('cueSpikeTrainHPCError')-1)
    cueErrorPFC = np.abs(rmat.get('cueSpikeTrainPFCError')-1)
    indAll = np.unravel_index(np.argmin(cueErrorAll,axis=None),cueErrorAll.shape)
    indHPC = np.unravel_index(np.argmin(cueErrorHPC,axis=None),cueErrorHPC.shape)
    indPFC = np.unravel_index(np.argmin(cueErrorPFC,axis=None),cueErrorPFC.shape)
    connDict.update({'modelValuesAll':[rangeThresholdAll[indAll[0]],rangeRemainAll[indAll[1]]],
                     'modelValuesHPC':[rangeThresholdHPC[indHPC[0]],rangeRemainHPC[indHPC[1]]],
                     'modelValuesPFC':[rangeThresholdPFC[indPFC[0]],rangeRemainPFC[indPFC[1]]],})
    return connDict

def loadValuesFromFileSpatial(fileOfInterest,justFileName):
    spaConnectionGraph,hpcInds,pfcInds = generateDirectedGraphsSpatialOnly(justFileName)    
    connDict = {'connectionAll':spaConnectionGraph,
                'connectionHPC':np.take(np.take(spaConnectionGraph,hpcInds,axis=1),hpcInds,axis=0),
                'connectionPFC':np.take(np.take(spaConnectionGraph,pfcInds,axis=1),pfcInds,axis=0),
               }
    rangeThresholdAll,rangeRemainAll = generateStepSizing(connDict['connectionAll'])
    rangeThresholdHPC,rangeRemainHPC = generateStepSizing(connDict['connectionHPC'])
    rangeThresholdPFC,rangeRemainPFC = generateStepSizing(connDict['connectionPFC'])
    rmat = spio.loadmat(fileOfInterest)
    spaErrorAll = np.abs(rmat.get('spaSpikeTrainError')-1)
    spaErrorHPC = np.abs(rmat.get('spaSpikeTrainHPCError')-1)
    spaErrorPFC = np.abs(rmat.get('spaSpikeTrainPFCError')-1)
    indAll = np.unravel_index(np.argmin(spaErrorAll,axis=None),spaErrorAll.shape)
    indHPC = np.unravel_index(np.argmin(spaErrorHPC,axis=None),spaErrorHPC.shape)
    indPFC = np.unravel_index(np.argmin(spaErrorPFC,axis=None),spaErrorPFC.shape)
    connDict.update({'modelValuesAll':[rangeThresholdAll[indAll[0]],rangeRemainAll[indAll[1]]],
                     'modelValuesHPC':[rangeThresholdHPC[indHPC[0]],rangeRemainHPC[indHPC[1]]],
                     'modelValuesPFC':[rangeThresholdPFC[indPFC[0]],rangeRemainPFC[indPFC[1]]],})
    return connDict

def pathWalk(allRawPaths,allSavPaths):
    for z,singleRawP in enumerate(allRawPaths):
        singleSavP = allSavPaths[z]
        for aRoot,aDirs,aFiles in os.walk(singleRawP):
            for aFile in aFiles:
                if aFile.endswith('.mat'):
                    currRawFile = os.path.join(singleRawP,aFile)
                    currSavFile = os.path.join(singleSavP,aFile)
                    if 'Cue/' in currRawFile:
                        dictToSave = loadValuesFromFileCue(currRawFile,aFile)
                    elif 'Spa/' in currRawFile:
                        dictToSave = loadValuesFromFileSpatial(currRawFile,aFile)
                    saveVar = spio.savemat(currSavFile,dictToSave)
        print(f'Built from {singleRawP}')
    return 'Done'

rawPs = []

savPs = []

warnings.filterwarnings('ignore')
doneVar = pathWalk(rawPs,savPs)
print(doneVar)


Built from /home/aditya/Extracted_Data/31_Biophysical_Sim/02_Sim_Error/Cue/
Built from /home/aditya/Extracted_Data/31_Biophysical_Sim/02_Sim_Error/Spatial/
Done
